# 1. Basic Classes

Python classes look familiar to Java developers but work very differently under the hood — no access modifiers, no compile-time type enforcement, and `self` must be passed **explicitly**.

This notebook covers: class definition, `self`, access conventions, class vs instance variables, `@classmethod`, `@staticmethod`, and dynamic attributes.

### 1.1 Defining a Class

**☕ JAVA:**
```java
public class Person {
    private String name;
    private int age;

    public Person(String name, int age) {
        this.name = name;   // 'this' is implicit
        this.age = age;
    }

    public String greet() {
        return "Hello, I'm " + name;
    }
}
```

**🐍 PYTHON:** `self` replaces `this` but is **explicitly required** as the first parameter of every instance method. The constructor is always called `__init__`.

In [2]:
class Person:
    """A simple Person class"""
    def __init__(self, name: str, age: int):
        """Constructor -always called __init__."""
        self.name = name
        self.age = age
    
    def greet(self) -> str:
        """Instance method - 'self' is required as first parameter"""
        return f"Hello, I'm {self.name} and i am {self.age} years old"
    
person_1 = Person("Alice", 30)
print(person_1.greet())
print(f"Name: {person_1.name}, Age: {person_1.age}")

Hello, I'm Alice and i am 30 years old
Name: Alice, Age: 30


In [3]:
person_1.name = "Bob"
print(person_1.greet()) 

Hello, I'm Bob and i am 30 years old


> ⚠️ **Biggest gotcha for Java devs:** Forgetting `self` as the first parameter will give you an error like `greet() takes 0 positional arguments but 1 was given`. Python passes the instance automatically, but you **must** declare `self` explicitly.

### 1.2 Access Modifiers — Conventions, Not Enforcement

**☕ JAVA:** Strict access control with `public`, `private`, `protected`, package-private.

**🐍 PYTHON:** No access modifiers at all! Only **conventions**:

| Convention | Meaning | Java Equivalent |
|-----------|---------|----------------|
| `self.name` | Public | `public` |
| `self._name` | "Private" by convention — use but don't abuse | `protected` |
| `self.__name` | Name mangling → becomes `_ClassName__name` | Closest to `private` |

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float):
        self.owner = owner # Public - anyone can have access
        self._balance = balance # Convention: "treat as private"
        self.__pin = 12345 # Name mangling: becomes _Account__pin

acc = Account("Alice", 100.0)
print(f"Public: {acc.owner}")
print(f"Treat it as private: {acc._balance}")
print(f"Name Mangling: {acc._Account__pin}")
# print(f"Treat as private: {acc.__pin}") # This give error


Public: Alice
Treat it as private: 100.0
Name Mangling: 12345


### 1.3 Class Variables vs Instance Variables

**☕ JAVA:**
```java
public class Dog {
    static String species = "Canis familiaris";  // shared
    private String name;                          // per-instance
}
```

**🐍 PYTHON:** Variables defined directly in the class body are class variables (shared). Variables set on `self` in `__init__` are instance variables.

In [5]:
class Dog:
    species = "Canis familiars" # Class variable (like static)
    count = 0 # Track instances

    def __init__(self, name: str):
        self.name = name # Instance variable
        Dog.count += 1 # Modify class variable

d1 = Dog("Buddy")
d2 = Dog("Rex")

print(f"Species: {Dog.species}") # Access via class
print(f"Species: {d1.species}")  # Access via instance
print(f"Total dogs: {Dog.count}")# Shared counter 

Species: Canis familiars
Species: Canis familiars
Total dogs: 2


### 1.4 `@classmethod` — Factory Methods

**☕ JAVA:**
```java
public static Person fromString(String data) {
    String[] parts = data.split(",");
    return new Person(parts[0].trim(), Integer.parseInt(parts[1].trim()));
}
```

**🐍 PYTHON:** `@classmethod` receives the **class itself** as `cls` (not `self`). This is better than Java's static factory because `cls` enables proper subclass creation — if a subclass calls the factory, `cls` refers to the subclass!

In [6]:
class Person:
    """A simple Person class"""
    def __init__(self, name: str, age: int):
        """Constructor -always called __init__."""
        self.name = name
        self.age = age

    @classmethod
    def from_string(cls, data: str) -> "Person":
        name, age = data.split(",")
        return cls(name.strip(), int(age.strip())) # cls - not Person!
    
    @classmethod
    def from_birth_year(cls, name: str, year_of_birth: int) -> "Person":
        from datetime import date
        return cls(name, date.today().year - year_of_birth)
    
p1 = Person.from_string("Alice, 30")
p2 = Person.from_birth_year("Bob", 1989) 

print(f"{p1.name} : {p1.age}")
print(f"{p2.name} : {p2.age}")

Alice : 30
Bob : 37


### 1.5 `@staticmethod` — Utility Methods

**☕ JAVA:** `static` methods are very common.

**🐍 PYTHON:** `@staticmethod` exists but is **rarely used** in practice. If you don't need `self` or `cls`, consider making it a plain module-level function instead — that's more Pythonic.

Use `@staticmethod` only when the method **logically belongs to the class** but doesn't need instance or class access.

In [7]:
class MathUtils:
    @staticmethod
    def is_even(n: int) -> bool:
        return n % 2 == 0

    @staticmethod
    def clamp(value: float, low: float, high: float) -> float:
        return max(low, min(high, value))
    
print(f"Is 4 even? {MathUtils.is_even(4)}")
print(f"Clamp 15 to [0, 10]: {MathUtils.clamp(15, 0 , 10)}")

Is 4 even? True
Clamp 15 to [0, 10]: 10


### 1.6 Dynamic Attributes — Classes Are Mutable!

**☕ JAVA:** Object structure is fixed at compile time. You can't add a field to an existing object.

**🐍 PYTHON:** You can add attributes to any object **at any time**. This is powerful but can lead to bugs.

In [8]:
class User:
    def __init__(self, name: str):
        self.name = name

user_1 = User("Alice")
print(user_1.name)

# Add dynamically -no error !
user_1.email = "alice@example.com"
user_1.role = "admin"

print(f"{user_1.name} - {user_1.email} - {user_1.role}")


Alice
Alice - alice@example.com - admin


In [9]:
print(f"All attributes: {user_1.__dict__}")

All attributes: {'name': 'Alice', 'email': 'alice@example.com', 'role': 'admin'}


### How we avoid to pass new attributes?


In [ ]:
class StrictUser:
    __slots__ = ("name", "age") # This make our class strict for the attributes

    def __init__(self, name: str, age: str):
        self.name = name
        self.age = age

strict_u = StrictUser("Bob", 45)
print(f"{strict_u.name} : {strict_u.age}")

try:
    strict_u.email = "bob@example.com"
except AttributeError as e:
    print(f"Error: {e}")

Bob : 45
Error: 'StrictUser' object has no attribute 'email'


---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Book` class with `title`, `author`, and `pages`. Add a `@classmethod` factory `from_string` that parses `"Title|Author|Pages"`.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `Counter` class that tracks how many instances have been created using a class variable. Each instance should have a unique `id`.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a class with `__slots__` set to `("x", "y")`. Verify that you cannot add a `z` attribute.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Constructor | Same name as class | `__init__` |
| Self reference | `this` (implicit) | `self` (explicit, **required**) |
| Access modifiers | `public`/`private`/`protected` | Conventions: `_` and `__` |
| True private? | Yes (`private`) | No — `__name` is mangled, not hidden |
| Class variable | `static` field | Defined in class body |
| Instance variable | Non-static field | Set on `self` in `__init__` |
| Factory method | `static` method returning `new` | `@classmethod` with `cls()` |
| Static method | `static` (very common) | `@staticmethod` (rarely used) |
| Dynamic attributes | Not possible | Can add anytime (`obj.x = ...`) |
| Lock structure | Always fixed | Use `__slots__` to restrict |